# test_react — ReAct Intent Router

Validates the classifier node correctly splits traffic:
- Conversational input -> chat node -> answer, no plan
- Research input -> planner -> executor loop -> synthesiser -> answer

Run each cell and inspect the output to confirm routing.

In [3]:
import sys
sys.path.insert(0, '../app/backend')

# Blank initial state matching server.py cold-start
BLANK = {
    'task': '',
    'messages': [],
    'plan': [],
    'status': '',  # empty -> supervisor -> classifier
    'results': [],
    'ui_events': [],
}

from graph import graph
print('Graph compiled OK')

Graph compiled OK


## Path 1: conversational input

Expected:
- `status == 'done'`
- `plan == []` (classifier skipped planner)
- `results == []` (executor never ran)
- `ui_events` has one entry `{type: 'answer', content: <str>}`

In [4]:
state_chat = {**BLANK, 'task': 'Hello, how are you?'}
result_chat = graph.invoke(state_chat)

print('status     :', result_chat['status'])
print('plan       :', result_chat['plan'])    # should be []
print('results    :', result_chat['results']) # should be []
print('ui_events  :', result_chat['ui_events'])

assert result_chat['status'] == 'done', 'Expected status=done'
assert result_chat['plan'] == [], 'Expected empty plan for chat path'
assert result_chat['results'] == [], 'Expected no executor results'
assert result_chat['ui_events'], 'Expected at least one ui_event'
assert result_chat['ui_events'][-1]['type'] == 'answer', 'Expected answer event'

print()
print('Answer:', result_chat['ui_events'][-1]['content'])
print()
print('PASS: chat path routed correctly')

status     : done
plan       : []
results    : []
ui_events  : [{'type': 'answer', 'content': "I'm doing well, thanks for asking! How about you?"}]

Answer: I'm doing well, thanks for asking! How about you?

PASS: chat path routed correctly


## Path 2: research input

Expected:
- `status == 'done'`
- `plan != []` (planner ran and produced steps)
- `results != []` (executor ran at least one step)
- `ui_events` has one entry `{type: 'answer', ...}` from synthesiser

In [8]:
state_research = {**BLANK, 'task': 'What is the capital of France and what is its population?'}
result_research = graph.invoke(state_research)

print('status     :', result_research['status'])
print('plan steps :', len(result_research['plan']), result_research['plan'])
print('results    :', len(result_research['results']), 'item(s)')
print('ui_events  :', result_research['ui_events'])

assert result_research['status'] == 'done', 'Expected status=done'
# plan may or may not be empty by the time graph is done (executor pops steps),
# but results must be non-empty to confirm executor ran.
assert result_research['results'], 'Expected executor results for research path'
assert result_research['ui_events'], 'Expected ui_event with answer'
assert result_research['ui_events'][-1]['type'] == 'answer'

print()
print('Answer:', result_research['ui_events'][-1]['content'][:300], '...')
print()
print('PASS: research path routed correctly')

status     : done
plan steps : 0 []
results    : 2 item(s)
ui_events  : [{'type': 'answer', 'content': '# The Capital of France and Its Population\n\nThe capital city of France is **Paris** [Finding 1]. While it is the undisputed political and cultural heart of the country, the number of people living there depends heavily on how you define the city\'s boundaries.\n\n### Defining the Population\nBecause Paris is surrounded by a vast urban region, different statistics provide different answers:\n\n*   **City Proper (Commune):** If you count only the official administrative limits of the city itself, the population is approximately **2.1 million** people [Finding 2].\n*   **Greater Paris (Agglomeration):** This includes the city proper plus the surrounding towns and suburbs that form a continuous built-up area. This figure is estimated at **11.4 million** people [Finding 2].\n*   **Metropolitan Area:** This is the broadest definition, encompassing the entire region known as Île-de-France

## Path 3: streaming — verify SSE events

Uses `graph.stream()` to inspect which node names appear in the chunks,
confirming classifier -> chat or classifier -> planner -> executor -> synthesiser.

In [9]:
print('--- Stream: conversational ---')
nodes_seen = []
for chunk in graph.stream({**BLANK, 'task': 'Hey FRIDAY!'}):
    node = list(chunk.keys())[0]
    nodes_seen.append(node)
    print(f'  chunk from: {node}')

assert 'classifier' in nodes_seen, 'classifier must appear'
assert 'chat' in nodes_seen, 'chat must appear for conversational'
assert 'planner' not in nodes_seen, 'planner must NOT run for conversational'
print('PASS')

--- Stream: conversational ---
  chunk from: classifier
  chunk from: chat
PASS


In [10]:
print('--- Stream: research ---')
nodes_seen = []
for chunk in graph.stream({**BLANK, 'task': 'Explain how neural networks learn using backpropagation.'}):
    node = list(chunk.keys())[0]
    nodes_seen.append(node)
    print(f'  chunk from: {node}')

assert 'classifier' in nodes_seen, 'classifier must appear'
assert 'planner' in nodes_seen, 'planner must run for research'
assert 'chat' not in nodes_seen, 'chat must NOT run for research'
print('PASS')

--- Stream: research ---
  chunk from: classifier
  chunk from: planner
  chunk from: executor
  chunk from: executor
  chunk from: executor
  chunk from: executor
  chunk from: executor
  chunk from: executor
  chunk from: executor
  chunk from: synthesiser
PASS
